# 02 - Michel Electron Mini-Analysis

Goal: build a detector-agnostic Michel electron candidate table from reconstructed SPINE particles, estimate simple selection metrics when truth is available, and send interesting entries to Spinal Tap.

Michel electrons are useful here because the exercise is high-level and physics-facing, but still forces us to reason about SPINE object fields: particle shape, interaction membership, particle size, deposited charge, endpoints, matching, and topology.

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from spine.driver import Driver

# Edit these two lines for the sample you want to inspect.
# DATA_PATH may be a single HDF5 file or a glob such as "/path/to/reco/*.h5".
DATA_PATH = "CHANGE_ME/reconstructed_spine_file.h5"

# Use None if no detector geometry is needed. Common examples:
# "icarus", "sbnd", "2x2", "nd-lar", "protodune-sp", "protodune-vd"
DETECTOR = None

CONFIG_CANDIDATES = [
    Path("../config/read_spine_hdf5.yaml"),
    Path("tutorials/config/read_spine_hdf5.yaml"),
]
CONFIG_PATH = next((path for path in CONFIG_CANDIDATES if path.exists()), None)

if DATA_PATH.startswith("CHANGE_ME"):
    raise RuntimeError(
        "Set DATA_PATH at the top of this notebook to the tutorial HDF5 file or file glob."
    )
if CONFIG_PATH is None:
    raise RuntimeError("Could not find tutorials/config/read_spine_hdf5.yaml")

cfg_text = CONFIG_PATH.read_text().replace("DATA_PATH", DATA_PATH)
cfg = yaml.safe_load(cfg_text)
if DETECTOR:
    cfg["geo"] = {"detector": DETECTOR}
driver = Driver(cfg)
print(f"Opened {DATA_PATH}")
print(f"Entries: {len(driver)}")
if DETECTOR:
    print(f"Detector geometry: {DETECTOR}")


## Analysis idea

A Michel electron is an electron from a stopped muon decay. In reconstructed SPINE objects we will look for:

1. a particle with Michel semantic shape;
2. a track-like particle in the same interaction;
3. small spatial separation between the Michel points and the candidate parent track;
4. enough reconstructed points to avoid tiny fragments.

This is not a final detector-specific Michel selection. It is a compact analysis skeleton that works across detectors as long as the HDF5 file contains reconstructed particles.

In [ ]:
N_ENTRIES = min(len(driver), 50)
print(f"Scanning {N_ENTRIES} entries")

In [ ]:
from spine.utils.globals import MICHL_SHP, TRACK_SHP, SHAPE_LABELS

print(f"Michel shape id: {MICHL_SHP} -> {SHAPE_LABELS[MICHL_SHP]}")
print(f"Track shape id:  {TRACK_SHP} -> {SHAPE_LABELS[TRACK_SHP]}")

## Pause on the distance function

The line `diff = a[:, None, :] - b[None, :, :]` builds all pairwise point differences between two particles. That gives us the closest approach between a Michel candidate and a track without relying on detector-specific geometry.

In [ ]:
def points(p):
    pts = getattr(p, "points", None)
    if pts is None:
        return np.empty((0, 3))
    return np.asarray(pts)

def min_point_distance(p1, p2):
    a = points(p1)
    b = points(p2)
    if len(a) == 0 or len(b) == 0:
        return np.inf
    diff = a[:, None, :] - b[None, :, :]
    return float(np.sqrt(np.sum(diff * diff, axis=-1)).min())

def deposition_sum(p):
    value = getattr(p, "depositions_sum", None)
    if value is not None:
        return float(value)
    depositions = getattr(p, "depositions", None)
    return float(np.sum(depositions)) if depositions is not None else np.nan

In [ ]:
MUON_MIN_POINTS = 20
ATTACH_THRESHOLD_CM = 3.0
MICHEL_MIN_POINTS = 5
MATCH_THRESHOLD = 0.1

print({
    "muon_min_points": MUON_MIN_POINTS,
    "attach_threshold_cm": ATTACH_THRESHOLD_CM,
    "michel_min_points": MICHEL_MIN_POINTS,
    "match_threshold": MATCH_THRESHOLD,
})

In [ ]:
def best_attached_track(candidate, particles):
    same_interaction_tracks = [
        p for p in particles
        if getattr(p, "interaction_id", -1) == getattr(candidate, "interaction_id", -2)
        and getattr(p, "shape", -1) == TRACK_SHP
        and getattr(p, "size", 0) >= MUON_MIN_POINTS
    ]
    if not same_interaction_tracks:
        return None, np.inf
    distances = [(track, min_point_distance(candidate, track)) for track in same_interaction_tracks]
    return min(distances, key=lambda item: item[1])

def best_truth_match(p):
    ids = list(getattr(p, "match_ids", []))
    overlaps = list(getattr(p, "match_overlaps", []))
    if not ids or not overlaps:
        return -1, 0.0
    index = int(np.argmax(overlaps))
    return ids[index], float(overlaps[index])

def michel_candidate_row(entry, p, particles):
    parent_track, attach_dist = best_attached_track(p, particles)
    match_id, match_overlap = best_truth_match(p)
    return {
        "entry": entry,
        "particle_id": getattr(p, "id", -1),
        "interaction_id": getattr(p, "interaction_id", -1),
        "size": getattr(p, "size", np.nan),
        "depositions_sum": deposition_sum(p),
        "is_contained": getattr(p, "is_contained", None),
        "attach_dist_cm": attach_dist,
        "attached_track_id": getattr(parent_track, "id", -1) if parent_track is not None else -1,
        "attached_track_size": getattr(parent_track, "size", np.nan) if parent_track is not None else np.nan,
        "match_id": match_id,
        "match_overlap": match_overlap,
        "is_truth_matched": match_overlap >= MATCH_THRESHOLD,
        "ke": getattr(p, "ke", np.nan),
    }

## Build the candidate table

Pause on `p.shape == MICHL_SHP`: this is where the high-level analysis becomes a SPINE-object query. Everything after that is ordinary table building.

In [ ]:
rows = []
true_rows = []

for entry in range(N_ENTRIES):
    data = driver.process(entry=entry)
    reco_particles = data.get("reco_particles", [])
    truth_particles = data.get("truth_particles", [])

    for p in reco_particles:
        if getattr(p, "shape", -1) == MICHL_SHP:
            rows.append(michel_candidate_row(entry, p, reco_particles))

    for tp in truth_particles:
        if getattr(tp, "shape", -1) == MICHL_SHP:
            match_id, match_overlap = best_truth_match(tp)
            true_rows.append({
                "entry": entry,
                "truth_particle_id": getattr(tp, "id", -1),
                "interaction_id": getattr(tp, "interaction_id", -1),
                "size": getattr(tp, "size", np.nan),
                "depositions_sum": deposition_sum(tp),
                "is_contained": getattr(tp, "is_contained", None),
                "match_id": match_id,
                "match_overlap": match_overlap,
                "is_reco_matched": match_overlap >= MATCH_THRESHOLD,
                "ke": getattr(tp, "ke", np.nan),
            })

candidate_columns = [
    "entry",
    "particle_id",
    "interaction_id",
    "size",
    "depositions_sum",
    "is_contained",
    "attach_dist_cm",
    "attached_track_id",
    "attached_track_size",
    "match_id",
    "match_overlap",
    "is_truth_matched",
    "ke",
]
truth_columns = [
    "entry",
    "truth_particle_id",
    "interaction_id",
    "size",
    "depositions_sum",
    "is_contained",
    "match_id",
    "match_overlap",
    "is_reco_matched",
    "ke",
]
candidates = pd.DataFrame(rows, columns=candidate_columns)
truth_michels = pd.DataFrame(true_rows, columns=truth_columns)

display(candidates.head())
display(truth_michels.head())

In [ ]:
if len(candidates):
    candidates["passes_michel_selection"] = (
        (candidates["size"] >= MICHEL_MIN_POINTS)
        & (candidates["attach_dist_cm"] <= ATTACH_THRESHOLD_CM)
    )
else:
    candidates["passes_michel_selection"] = []

summary = candidates.groupby("passes_michel_selection").size().rename("n_reco_michel_candidates").to_frame()
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
if len(candidates):
    candidates["size"].hist(ax=axes[0], bins=30)
    candidates["attach_dist_cm"].replace(np.inf, np.nan).hist(ax=axes[1], bins=30)
    candidates["depositions_sum"].hist(ax=axes[2], bins=30)
axes[0].set_xlabel("Michel candidate points")
axes[1].set_xlabel("closest track distance [cm]")
axes[2].set_xlabel("candidate deposition sum")
plt.tight_layout()

In [ ]:
selected = candidates[candidates["passes_michel_selection"]].copy()
selected.head(20)

## Truth metrics, if truth is available

These are intentionally simple counts. They are meant to start a discussion, not to certify a production performance number.

In [ ]:
n_pred = int(candidates["passes_michel_selection"].sum()) if len(candidates) else 0
n_pred_matched = int((candidates["passes_michel_selection"] & candidates["is_truth_matched"]).sum()) if len(candidates) else 0
n_true = len(truth_michels)
n_true_matched = int(truth_michels["is_reco_matched"].sum()) if len(truth_michels) else 0

purity = n_pred_matched / n_pred if n_pred else np.nan
efficiency = n_true_matched / n_true if n_true else np.nan

pd.DataFrame([{
    "selected_reco_michels": n_pred,
    "selected_truth_matched": n_pred_matched,
    "true_michels": n_true,
    "true_reco_matched": n_true_matched,
    "rough_purity": purity,
    "rough_efficiency": efficiency,
}])

## Spinal Tap handoff

The useful debugging output is a small list of selected candidates and suspicious failures:

- selected Michel candidates with large attachment distance;
- selected candidates with no truth match;
- true Michels with no reco match;
- high-charge candidates that fail the attachment cut.

In [ ]:
event_list = candidates.sort_values(
    ["passes_michel_selection", "is_truth_matched", "entry", "particle_id"],
    ascending=[False, True, True, True],
)
event_list[[
    "entry",
    "particle_id",
    "interaction_id",
    "passes_michel_selection",
    "is_truth_matched",
    "size",
    "attach_dist_cm",
    "attached_track_id",
    "depositions_sum",
]].to_csv("spinal_tap_michel_event_list.csv", index=False)
print("Wrote spinal_tap_michel_event_list.csv")
event_list.head(20)

## Live exercise

Choose one:

1. Change `ATTACH_THRESHOLD_CM` from 3 cm to 1 cm or 5 cm. What happens to the candidate count?
2. Raise `MICHEL_MIN_POINTS`. Which candidates disappear first?
3. Open one unmatched selected candidate in Spinal Tap. Is it a real Michel, a delta ray, a fragment, or a shower piece?
4. Open one true unmatched Michel in Spinal Tap. Was it missed by segmentation, clustering, or matching?

## Offline extensions

- Add a parent-muon stopping requirement using the track endpoint and nearby charge.
- Estimate a rough Michel energy spectrum from `depositions_sum` or `ke`, then compare matched reco and truth.
- Split efficiency by candidate size, containment, or detector region.
- Build a curated Spinal Tap list of five clean Michels and five failure modes.
- Try the same notebook on a second detector sample and identify which thresholds need retuning.